# Officer Performance Analysis

## Objective

This notebook evaluates the performance of recovery officers by analyzing the number of claims handled, recovery amounts, collected amounts, outstanding balances, and collection efficiency.

### Business Questions

- Which officer handles the most claims?
- Which officer recovers the most money?
- Which officer collects the highest amount?
- Which officer has the highest collection efficiency?
- Which officers may require operational attention?

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from src.preprocessing import prepare_data
pd.set_option("display.max_columns", None)

In [2]:
df= prepare_data()

### Q1) Which officer handles the most claims?

In [3]:
claims_per_officer = (
    df.groupby("Officer")
      .agg(
          Claims=("Claim ID", "count")
      )
      .sort_values("Claims", ascending=False)
      .reset_index()
)

claims_per_officer

,Officer,Claims
0,MAJED,1130
1,Munirah,680
2,Jasim,254
3,Sultan Alharbi,132
4,NAIF,111
5,RIYADH MOHAMMED ALI ALBATI,90
6,WALEED MOHAMMED GHUBAYN,88
7,Maha Mohammed,80
8,M.ALBABTAIN,60
9,Madeline,21


In [4]:
fig = px.bar(
    claims_per_officer,
    x="Claims",
    y="Officer",
    orientation="h",
    color="Claims",
    title="Number of Claims Handled by Each Officer",
    text_auto=".2s"
)

fig.update_layout(yaxis={'categoryorder':'total ascending'})

fig.show()

### Insight
This visualization highlights workload distribution among recovery officers.
Officers handling significantly larger numbers of claims may require workload balancing or additional support.

### Q2) Which officer manages the highest recovery amount?

In [5]:
recovery_per_officer = (
    df.groupby("Officer")
      .agg(
          Recovery=("Recovery Amount", "sum")
      )
      .sort_values("Recovery", ascending=False)
      .reset_index()
)

recovery_per_officer

,Officer,Recovery
0,MAJED,14172495.11
1,Munirah,5603985.54
2,Jasim,1587413.02
3,NAIF,1152169.78
4,Sultan Alharbi,937928.85
5,RIYADH MOHAMMED ALI ALBATI,613293.00
6,WALEED MOHAMMED GHUBAYN,587122.00
7,Maha Mohammed,562458.47
8,M.ALBABTAIN,368484.75
9,Madeline,277024.10


In [6]:
fig = px.bar(
    recovery_per_officer,
    x="Recovery",
    y="Officer",
    orientation="h",
    color="Recovery",
    title="Total Recovery Amount by Each Officer",
    text_auto=".2s"
)

fig.update_layout(yaxis={'categoryorder':'total ascending'})

fig.show()

### Insight
This chart identifies officers responsible for the largest recovery portfolios.
Higher recovery amounts do not necessarily indicate better performance and should be evaluated alongside collection efficiency.

### Q3) Which officer has collected the most money?

In [7]:
collected_per_officer = (
    df.groupby("Officer")
      .agg(
          Collected=("Collected Amount", "sum")
      )
      .sort_values("Collected", ascending=False)
      .reset_index()
)

collected_per_officer

,Officer,Collected
0,MAJED,892067.24
1,Munirah,248742.94
2,Jasim,121551.90
3,Madeline,107293.00
4,Sultan Alharbi,36518.90
5,Maha Mohammed,17718.84
6,M.ALBABTAIN,16757.75
7,WALEED MOHAMMED GHUBAYN,11461.00
8,NAIF,8989.45
9,RIYADH MOHAMMED ALI ALBATI,8158.00


In [8]:
fig= px.bar(
    collected_per_officer,
    x="Collected",
    y="Officer",
    orientation="h",
    color="Collected",
    title="Total Collected Amount by Each Officer",
    text_auto=".2s"
)

fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

### Q4) Which officer has the highest collection efficiency?

In [9]:
officer_summary = (
    claims_per_officer
    .merge(recovery_per_officer,on="Officer")
    .merge(collected_per_officer,on="Officer")
)

officer_summary["Remaining"] = (
    officer_summary["Recovery"] -
    officer_summary["Collected"]
)

officer_summary["Collection Rate"] = (
    officer_summary["Collected"] /
    officer_summary["Recovery"] * 100
).round(2)

officer_summary["Avg Recovery per Claim"] = (
    officer_summary["Recovery"] /
    officer_summary["Claims"]
).round(2)


officer_summary.sort_values(
    "Collection Rate",
    ascending=False,
    inplace=True
)

officer_summary

,Officer,Claims,Recovery,Collected,Remaining,Collection Rate,Avg Recovery per Claim
9,Madeline,21,277024.10,107293.00,169731.10,38.73,13191.62
2,Jasim,254,1587413.02,121551.90,1465861.12,7.66,6249.66
10,AYOUB,8,98206.95,7176.37,91030.58,7.31,12275.87
0,MAJED,1130,14172495.11,892067.24,13280427.87,6.29,12542.03
8,M.ALBABTAIN,60,368484.75,16757.75,351727.00,4.55,6141.41
1,Munirah,680,5603985.54,248742.94,5355242.60,4.44,8241.16
3,Sultan Alharbi,132,937928.85,36518.90,901409.95,3.89,7105.52
7,Maha Mohammed,80,562458.47,17718.84,544739.63,3.15,7030.73
6,WALEED MOHAMMED GHUBAYN,88,587122.00,11461.00,575661.00,1.95,6671.84
5,RIYADH MOHAMMED ALI ALBATI,90,613293.00,8158.00,605135.00,1.33,6814.37


In [10]:
fig = px.bar(
    officer_summary,
    x="Collection Rate",
    y="Officer",
    orientation="h",
    color="Collection Rate",
    title="Officer Collection Efficiency (%)",
    text_auto="Collection Rate",
    color_continuous_scale="Viridis"
)

fig.update_layout(yaxis={'categoryorder':'total ascending'})

fig.show()

### Insight
Collection efficiency compares the amount collected against the total recovery amount assigned to each officer.

This metric provides a fair performance comparison regardless of portfolio size and can be used to identify high-performing officers as well as those requiring operational support.

In [12]:
best = officer_summary.iloc[0]

print(f"""
🏆 Top Performing Officer

Officer : {best['Officer']}

Collection Rate : {best['Collection Rate']} %

Recovery Portfolio : {best['Recovery']:,.0f} SAR

Collected : {best['Collected']:,.0f} SAR

Average Recovery per Claim : {best['Avg Recovery per Claim']:,.0f} SAR

""")


🏆 Top Performing Officer

Officer : Madeline

Collection Rate : 38.73 %

Recovery Portfolio : 277,024 SAR

Collected : 107,293 SAR

Average Recovery per Claim : 13,192 SAR


